# Marine Cloud Brightening TEA: uncertainty dashboard
Companion to Tito Jankowski's [mcb-tea-cost-benefit](https://github.com/titojankowski/mcb-tea-cost-benefit) workbook.

**Tito's workbook is the engine and is never modified.** This notebook recalculates his own formulas (pycel), lets you move his inputs
across literature ranges, and shows how the key outputs respond: a ranked tornado, a Monte Carlo distribution (P10 / P50 / P90),
sweeps of the disputed inputs, and where the literature suggests a better method than a workbook formula (shown *next to* his number, never instead of it).
Every range and every computed quantity shows its origin: `Tito workbook <sheet>!<cell>: <formula>` or `literature: <paper, DOI, section>`.

Run the cells top to bottom (Runtime > Run all). Sections: A setup, B inputs, C uncertainty, D results and charts, E literature models, F sources and conflicts, G downloads.

## A. Setup

In [ ]:
%pip install -q openpyxl pycel pandas numpy matplotlib ipywidgets

In [ ]:
import os, io, csv, json, math, re, shutil, time, urllib.parse, urllib.request
REPO = "retospect/mcb-tea-cost-benefit"   # the PR commit flips these two lines to
BRANCH = "colab-notebook"                 #   REPO = "titojankowski/mcb-tea-cost-benefit", BRANCH = "main"
BASE = os.environ.get("MCB_BASE") or f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/"   # MCB_BASE: test hook
FILE = "Draft MCB TEA and Cost Benefit TJ 10122026.xlsx"
XLSX = "mcb_tea_original.xlsx"
def fetch(rel, dst):
    local = os.environ.get("MCB_DIR")                 # optional: a local checkout, for offline testing
    if local: shutil.copy(os.path.join(local, rel), dst)
    else: urllib.request.urlretrieve(BASE + urllib.parse.quote(rel), dst)
fetch(FILE, XLSX)
for f in ("ranges.csv", "survey.csv", "summary.csv", "sources.csv", "literature_models.csv", "annotate_xlsx.py"): fetch("colab/" + f, f)
URL = BASE + urllib.parse.quote(FILE)
from IPython.display import HTML, display
print("workbook:", URL if not os.environ.get("MCB_DIR") else "local copy", os.path.getsize(XLSX), "bytes (Tito's file, read-only here)")
display(HTML(f'<a href="{URL}" download>Download Tito\'s workbook (xlsx)</a>'))

**The engine.** `openpyxl` reads the formulas, `pycel` evaluates them. Before anything else we check that pycel reproduces every value Excel saved in Tito's file.
When an input changes, pycel recalculates only the cells that depend on it (the dependency cone of the outputs we ask for), which is why a recalculation takes milliseconds.

In [ ]:
import numpy as np, pandas as pd, openpyxl
from pycel import ExcelCompiler
wbf = openpyxl.load_workbook(XLSX)                      # formulas
wbv = openpyxl.load_workbook(XLSX, data_only=True)      # values Excel saved
formula_cells = [(s.title, c.coordinate) for s in wbf for r in s.iter_rows() for c in r if c.data_type == "f"]
def agree(a, b):
    if isinstance(a, (int, float)) and isinstance(b, (int, float)): return math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9)
    return a == b
def new_engine(): return ExcelCompiler(filename=XLSX)
xl0 = new_engine(); bad = []
t0 = time.time()
for s, a in formula_cells:
    try: got = xl0.evaluate(f"'{s}'!{a}")
    except Exception as e: got = f"ERROR {type(e).__name__}"
    if not agree(got, wbv[s][a].value): bad.append((s, a, got, wbv[s][a].value))
print(f"pycel reproduces {len(formula_cells) - len(bad)} of {len(formula_cells)} formula cells in {time.time() - t0:.1f} s (tolerance 1e-6 relative).")
for b in bad[:10]: print("  mismatch", b)

## B. Inputs
Inputs are grouped by Tito's tabs. Each slider spans the range in `ranges.csv` (low / **Tito's value** / high). Green = literature note, **red = conflict** between the workbook and the literature,
grey = origin of the range. DOIs are links. Use the preset menu for Tito's values, the literature-adjusted central, or the pessimistic / optimistic ends (chosen per input by their effect on the benefit-cost ratio of tab 6b).

In [ ]:
import ipywidgets as w
R = pd.read_csv("ranges.csv", dtype=str, keep_default_na=False)
for c in ("low", "central", "high"): R[c] = R[c].astype(float)
R["key"] = list(zip(R.sheet, R.cell))
for k, c in zip(R.key, R.central):
    cur = wbv[k[0]][k[1]].value
    if not math.isclose(cur, c, rel_tol=1e-9): print("WARNING central differs from workbook:", k, cur, c)
INP = list(R.key)                                   # the uncertain inputs, in file order
CENTRAL = dict(zip(R.key, R.central))
S3A, S4, S5A, S6A, S6B, S6C, S1, S2 = ("3a. Outputs - Dedicated Vessel", "4. Scenario Comparison", "5a. Cooling Model - Local", "6a. Cooling Model - Global",
    "6b. Benefit - Global - Vessel", "6c. Benefit - Global Piggyback", "1. Technology Description", "2. Cost Inputs")
OUT = [  # label, sheet, cell, unit, shown in tornado
 ("3a levelized cost, dedicated vessel", S3A, "C13", "$/t aerosol", True), ("3a annual cost, dedicated vessel", S3A, "B13", "$/yr", False),
 ("4 levelized cost, ship piggyback (3b)", S4, "B7", "$/t aerosol", True), ("4 levelized cost, fixed platform (3c)", S4, "B8", "$/t aerosol", False),
 ("4 levelized cost, stationary vessel (3d)", S4, "B9", "$/t aerosol", False), ("5a cost of cooling produced (local)", S5A, "B34", "$/degC-ha-day", True),
 ("6b cost of 1 K per year, E3SMv2, dedicated", S6B, "E12", "$/yr", False),
 ("6b benefit-cost, CESM2", S6B, "D29", "ratio", False), ("6b benefit-cost, CESM2-Ghosh", S6B, "D30", "ratio", False),
 ("6b benefit-cost, E3SMv2", S6B, "D31", "ratio", True), ("6b benefit-cost, UKESM1.0", S6B, "D32", "ratio", False),
 ("6c benefit-cost, CESM2 (piggyback)", S6C, "D38", "ratio", False), ("6c benefit-cost, CESM2-Ghosh", S6C, "D39", "ratio", False),
 ("6c benefit-cost, E3SMv2 (piggyback)", S6C, "D40", "ratio", True), ("6c benefit-cost, UKESM1.0", S6C, "D41", "ratio", False)]
OKEY = [(o[1], o[2]) for o in OUT]
def origin_of(sheet, cell): return f"Tito workbook {sheet.split('.')[0]}!{cell}: {wbf[sheet][cell].value}"
class Engine:
    def __init__(self): self.xl = new_engine(); [self.xl.evaluate(f"'{s}'!{a}") for s, a in OKEY]
    def run(self, vals, extra=None):
        for k, v in vals.items(): self.xl.set_value(f"'{k[0]}'!{k[1]}", float(v))
        for k, v in (extra or {}).items(): self.xl.set_value(f"'{k[0]}'!{k[1]}", float(v))
        return np.array([self.xl.evaluate(f"'{s}'!{a}") for s, a in OKEY], dtype=float)
E = Engine()
tito = E.run(CENTRAL)
assert all(math.isclose(a, wbv[s][c].value, rel_tol=1e-6) for a, (s, c) in zip(tito, OKEY)), "engine differs from Excel's saved values"
t0 = time.time(); N = 200
for _ in range(N): E.run({**CENTRAL, INP[0]: CENTRAL[INP[0]] * (1 + 0.01 * (_ % 2))})
RUN_S = (time.time() - t0) / N
L1_G_DAY = 0.41 * 1.03 * 0.030 * 86400 * 0.91            # literature mass model L1 (see section E)
print(f"One recalculation (set {len(INP)} inputs, evaluate {len(OKEY)} outputs): {RUN_S * 1000:.1f} ms")
# direction of each input on the benefit-cost ratio (6b, E3SMv2), used by the pessimistic / optimistic presets
iB = [i for i, o in enumerate(OUT) if o[2] == "D31" and o[1] == S6B][0]
pess, opt = {}, {}
for k in INP:
    r = R[R.key == k].iloc[0]; lo = E.run({**CENTRAL, k: r.low})[iB]; hi = E.run({**CENTRAL, k: r.high})[iB]
    pess[k], opt[k] = (r.low, r.high) if lo < hi else (r.high, r.low)
E.run(CENTRAL)
PRESETS = {"Tito's values": CENTRAL, "Literature-adjusted central": {**CENTRAL, (S1, "B9"): 0.645}, "Pessimistic (low benefit-cost)": pess, "Optimistic (high benefit-cost)": opt}

In [ ]:
def loglike(r): return r.low > 0 and r.high / r.low >= 20
sliders, rows_ui = {}, []
def link(d): return f'<a href="https://doi.org/{d}" target="_blank">doi:{d}</a>'
GREEN, RED, GREY = "#3a9a4a", "#e0453a", "#8a8a85"
for sheet, g in R.groupby("sheet", sort=False):
    rows_ui.append(w.HTML(f"<h4 style='margin:14px 0 4px'>Tito's tab: {sheet}</h4>"))
    for _, r in g.iterrows():
        if loglike(r):
            sl = w.FloatLogSlider(value=r.central, base=10, min=math.log10(r.low), max=math.log10(r.high), step=0.01, readout_format=".4g", continuous_update=False, layout=w.Layout(width="330px"))
        elif r.high > r.low:
            sl = w.FloatSlider(value=r.central, min=r.low, max=r.high, step=(r.high - r.low) / 200, readout_format=".4g", continuous_update=False, layout=w.Layout(width="330px"))
        else:
            sl = w.FloatText(value=r.central, layout=w.Layout(width="120px"))
        sliders[r.key] = sl
        info = (f"<b>{r.label}</b> <span style='color:{GREY}'>[{r.cell}, {r.unit}]</span><br>"
                f"low {r.low:.4g} / <b>Tito {r.central:.4g}</b> / high {r.high:.4g}<br>")
        if r.conflict: info += f"<span style='color:{RED};font-weight:bold'>{r.conflict}</span><br>"
        info += f"<span style='color:{GREEN}'>{r.note}</span> " + " ".join(link(d) for d in r.source_dois.split()) + f"<br><span style='color:{GREY};font-style:italic'>origin: {r.origin}</span>"
        rows_ui.append(w.HBox([w.HTML(info, layout=w.Layout(width="760px")), sl], layout=w.Layout(border_bottom="1px solid #8884", padding="4px 0")))
preset = w.Dropdown(options=list(PRESETS), value="Tito's values", description="Preset", layout=w.Layout(width="420px"))
def apply_preset(_=None):
    for k, v in PRESETS[preset.value].items(): sliders[k].value = v
preset.observe(apply_preset, names="value")
for key, v in json.loads(os.environ.get("MCB_EDITS", "{}")).items():     # automated test hook, e.g. {"2. Cost Inputs|B7": 0.12}
    sliders[tuple(key.split("|"))].value = v
display(w.VBox([preset] + rows_ui))

## C. Uncertainty
**Tornado**: each input is moved alone to its low and high value while the others stay at Tito's values; bars show the change of each key output, ranked by swing.
**Monte Carlo**: every input is drawn independently from a triangular distribution (low, Tito's value as the mode, high), N = 3,000 runs, to give P10 / P50 / P90.
**The particle-size input is held at Tito's 150 nm in the Monte Carlo** (`MC_VARY_SIZE = False`): in his formula mass per particle scales with d^3, so letting size vary to 49.5 nm multiplies cost per tonne by about 28 and would swamp every other input, while the literature mass model (section E) says the salt mass out does not depend on that size. The tornado and the size sweep still show it.
Ranges are the workbook's own brackets unless the origin column says literature. Independent draws ignore correlations, and the damage-per-degree input spans 0.6-12 % of GDP, so the benefit-cost spread is very wide.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb", "text.color": "#0b0b0b", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "axes.edgecolor": "#c9c8c2", "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "legend.frameon": False})
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"                # categorical slots 1-3 of the reference palette
lab = {k: R[R.key == k].iloc[0].label for k in INP}
# --- tornado
t0 = time.time(); TOR = {}
for i, o in enumerate(OUT):
    if not o[4]: continue
    rows = []
    for k in INP:
        r = R[R.key == k].iloc[0]
        vlo = E.run({**CENTRAL, k: r.low})[i]; vhi = E.run({**CENTRAL, k: r.high})[i]
        rows.append((lab[k], 100 * (vlo / tito[i] - 1), 100 * (vhi / tito[i] - 1), k))
    TOR[i] = sorted(rows, key=lambda x: -max(abs(x[1]), abs(x[2])))
E.run(CENTRAL)
print(f"tornado: {len(INP) * 2 * len(TOR)} recalculations in {time.time() - t0:.1f} s")
tl = [i for i in TOR]
fig, axs = plt.subplots(2, 3, figsize=(19, 9)); axs = axs.ravel()
for ax, i in zip(axs, tl):
    rows = TOR[i][:10][::-1]; y = np.arange(len(rows))
    ax.barh(y, [r[1] for r in rows], height=0.6, color=BLUE, label="input at its LOW value")
    ax.barh(y, [r[2] for r in rows], height=0.6, color=ORANGE, label="input at its HIGH value", alpha=0.9)
    ax.set_yticks(y); ax.set_yticklabels([r[0][:46] for r in rows], fontsize=8.5); ax.axvline(0, color="#0b0b0b", lw=1)
    ax.set_xlabel("change vs Tito's value (%)"); ax.set_title(f"{OUT[i][0]}\nTito: {tito[i]:,.4g} {OUT[i][3]}", fontsize=10)
axs[0].legend(loc="lower right", fontsize=8)
for ax in axs[len(tl):]: ax.axis("off")
fig.tight_layout(); plt.show()
print("Origin of each plotted output:"); [print("  ", OUT[i][0], "->", origin_of(OUT[i][1], OUT[i][2])) for i in tl]

In [ ]:
rng = np.random.default_rng(20261024); NMC = 3000
MC_VARY_SIZE = False                       # True: also sample the particle-size input (Tito formula, d^3)
lo, ce, hi = (np.array([R[c].iloc[j] for j in range(len(R))]) for c in ("low", "central", "high"))
t0 = time.time(); S = np.empty((NMC, len(OUT))); X = np.empty((NMC, len(INP)))
for n in range(NMC):
    x = np.where(hi > lo, rng.triangular(lo, ce, np.where(hi > lo, hi, lo + 1)), ce)
    if not MC_VARY_SIZE: x[INP.index((S1, "B10"))] = CENTRAL[(S1, "B10")]
    X[n] = x
    S[n] = E.run(dict(zip(INP, x)))
E.run(CENTRAL); MC_S = time.time() - t0
P = np.percentile(S, [10, 50, 90], axis=0)
print(f"Monte Carlo: {NMC} runs in {MC_S:.1f} s = {1000 * MC_S / NMC:.1f} ms per run (each run sets {len(INP)} inputs and re-evaluates {len(OUT)} outputs)")

## D. Key results and charts
Move the sliders in section B (or pick a preset), then press **Recalculate**. The table compares Tito's saved value, your current inputs, and the Monte Carlo P10 / P50 / P90 over the full ranges.
Each output's origin is the workbook formula that computes it.

In [ ]:
out_area = w.Output(); btn = w.Button(description="Recalculate", button_style="primary")
def current_vals(): return {k: s.value for k, s in sliders.items()}
def render(_=None):
    vals = current_vals(); cur = E.run(vals)
    df = pd.DataFrame({"output": [o[0] for o in OUT], "unit": [o[3] for o in OUT], "Tito (saved)": tito, "current inputs": cur, "change %": 100 * (cur / tito - 1),
                       "MC P10": P[0], "MC P50": P[1], "MC P90": P[2], "origin": [origin_of(o[1], o[2]) for o in OUT]})
    with out_area:
        out_area.clear_output(wait=True)
        changed = {lab[k]: v for k, v in vals.items() if not math.isclose(v, CENTRAL[k], rel_tol=1e-9)}
        print("Inputs changed from Tito's:", changed or "none")
        display(df.style.format({"Tito (saved)": "{:,.4g}", "current inputs": "{:,.4g}", "change %": "{:+.1f}", "MC P10": "{:,.4g}", "MC P50": "{:,.4g}", "MC P90": "{:,.4g}"}).hide(axis="index")
                .set_properties(subset=["origin"], **{"color": "#8a8a85", "font-style": "italic"}))
        fig, axs = plt.subplots(2, 3, figsize=(17, 9)); ax = axs.ravel()
        # 1 scenario comparison
        sc = [(S4, "B6", "3a dedicated vessel"), (S4, "B7", "3b ship piggyback"), (S4, "B8", "3c fixed platform"), (S4, "B9", "3d stationary vessel")]
        tv = [wbv[s][c].value for s, c, _ in sc]; cv = [E.xl.evaluate(f"'{s}'!{c}") for s, c, _ in sc]
        x = np.arange(4); ax[0].bar(x - 0.2, tv, 0.38, color=BLUE, label="Tito (saved)"); ax[0].bar(x + 0.2, cv, 0.38, color=ORANGE, label="current inputs")
        ax[0].set_xticks(x); ax[0].set_xticklabels([n for _, _, n in sc], rotation=15, fontsize=8.5); ax[0].set_ylabel("$/t aerosol"); ax[0].set_title("Scenario comparison (tab 4)"); ax[0].legend()
        # 2 stacked cost breakdown, 3a
        comp = [("B6", "Annualized capital"), ("B7", "Energy"), ("B8", "Seawater pumping"), ("B10", "Crew"), ("B11", "Maintenance + insurance"), ("B12", "Vessel energy")]
        cols = [BLUE, ORANGE, AQUA, "#eda100", "#e87ba4", "#6250d6"]
        tb = [wbv[S3A][c.replace("B", "C")].value for c, _ in comp]; cb = [E.xl.evaluate(f"'{S3A}'!{c.replace('B', 'C')}") for c, _ in comp]
        btm = np.zeros(2)
        for (c, nm), t_, c_, colr in zip(comp, tb, cb, cols):
            ax[1].bar([0, 1], [t_, c_], 0.55, bottom=btm, color=colr, label=nm, edgecolor="#fcfcfb", linewidth=2); btm += [t_, c_]
        ax[1].set_xticks([0, 1]); ax[1].set_xticklabels(["Tito (saved)", "current inputs"]); ax[1].set_ylabel("$/t aerosol"); ax[1].set_title("Cost breakdown, dedicated vessel (3a)"); ax[1].set_ylim(0, btm.max() * 1.45); ax[1].legend(fontsize=8, loc="upper center", ncol=3)
        # 3 CDF of 6b/6c BCR
        for idx, colr, nm in ((iB, BLUE, "6b dedicated"), (OKEY.index((S6C, "D40")), ORANGE, "6c piggyback")):
            v = np.sort(S[:, idx]); ax[2].plot(v, np.arange(1, len(v) + 1) / len(v), color=colr, lw=2, label=f"{nm} (E3SMv2)")
            ax[2].axvline(tito[idx], color=colr, ls="--", lw=1.2); ax[2].plot([cur[idx]], [np.interp(cur[idx], v, np.arange(1, len(v) + 1) / len(v))], "o", color=colr, ms=8, mec="#fcfcfb", mew=2)
        ax[2].axvline(1, color="#0b0b0b", lw=0.8); ax[2].set_xscale("log"); ax[2].set_xlabel("benefit-cost ratio (log scale)")
        from matplotlib.lines import Line2D
        h, _ = ax[2].get_legend_handles_labels()
        ax[2].set_ylabel("cumulative probability"); ax[2].set_title("Monte Carlo distribution (CDF)")
        ax[2].legend(handles=h + [Line2D([0], [0], color="#52514e", ls="--", lw=1.2, label="Tito (saved)"), Line2D([0], [0], marker="o", color="#52514e", ls="", ms=7, label="current inputs"), Line2D([0], [0], color="#0b0b0b", lw=0.8, label="ratio = 1")], fontsize=8, loc="lower right")
        # sweeps (others at the current slider values)
        base = dict(vals)
        szs = np.linspace(30, 150, 25); ys = [E.run({**base, (S1, "B10"): s_})[0] for s_ in szs]
        ax[3].plot(szs, ys, color=BLUE, lw=2, label="Tito's formula (mass ~ d^3)"); ax[3].set_yscale("log")
        E2 = Engine()
        ax[3].axhline(E2.run(base, extra={(S1, "B13"): L1_G_DAY})[0], color=AQUA, lw=2, ls="-.", label="literature mass model (L1)")
        ax[3].axvline(49.5, color=ORANGE, lw=2); ax[3].text(51, ax[3].get_ylim()[1] * 0.5, "Harrison 2025\nmedian 49.5 nm", color="#0b0b0b", fontsize=8.5, va="top")
        ax[3].axvline(150, color="#0b0b0b", ls="--", lw=1); ax[3].set_xlabel("dry particle diameter (nm)"); ax[3].set_ylabel("3a $/t aerosol"); ax[3].set_title("Disputed input: particle size"); ax[3].legend(fontsize=8)
        sv = np.linspace(0.05, 0.6, 24); b1 = [E.run({**base, (S6A, "B32"): s_})[iB] for s_ in sv]; b2 = [E.run({**base, (S6A, "B32"): s_})[OKEY.index((S6C, "D40"))] for s_ in sv]
        ax[4].plot(sv, b1, color=BLUE, lw=2, label="6b dedicated"); ax[4].plot(sv, b2, color=ORANGE, lw=2, label="6c piggyback"); ax[4].axhline(1, color="#0b0b0b", lw=0.8)
        ax[4].axvline(0.3, color="#0b0b0b", ls="--", lw=1); ax[4].set_yscale("log"); ax[4].set_xlabel("plume survival to cloud base (no literature value)"); ax[4].set_ylabel("benefit-cost, E3SMv2")
        ax[4].set_title("Disputed input: survival"); ax[4].legend()
        pw = np.linspace(25, 800, 24); c1 = [E.run({**base, (S1, "B17"): p_})[0] for p_ in pw]; c2 = [E.run({**base, (S1, "B17"): p_})[2] for p_ in pw]
        ax[5].plot(pw, c1, color=BLUE, lw=2, label="3a dedicated"); ax[5].plot(pw, c2, color=ORANGE, lw=2, label="3b piggyback"); ax[5].axvline(512, color="#0b0b0b", ls="--", lw=1)
        ax[5].axvline(513, color=AQUA, lw=2, alpha=0.7); ax[5].text(530, max(c1) * 0.9, "Tito 512 W =\nHarrison 513 W", fontsize=8.5, va="top")
        ax[5].set_xlabel("nozzle power (W)"); ax[5].set_ylabel("$/t aerosol"); ax[5].set_title("Input check: nozzle power"); ax[5].legend()
        fig.tight_layout(); plt.show()
        print("Origin of each chart: scenario bars = Tito workbook 4!B6:B9 (" + "; ".join(str(wbf[S4][f"B{r}"].value) for r in range(6, 10)) + "); cost breakdown = Tito workbook 3a!C6:C12 (" + wbf[S3A]["C6"].value + " etc.);")
        print("CDF = Monte Carlo through Tito's formulas for 6b!D31 (" + wbf[S6B]["D31"].value + ") and 6c!D40 (" + wbf[S6C]["D40"].value + "); sweeps = Tito's formulas with one input moved; the dash-dot line is the literature mass model L1 (section E).")
        E.run(vals)
btn.on_click(render)
display(w.VBox([btn, out_area]))
render()

## E. Better methods from the literature (optional, shown next to Tito's number)
Where a held paper gives a better method than a workbook formula, it is evaluated here **through Tito's own chain** and shown beside his value with the delta. Nothing replaces his formula.
Only methods and numbers read in the full text are used; `literature_models.csv` gives the paper, DOI and section for each. If no usable equation was found (survival to cloud base), the row says so.

In [ ]:
LM = pd.read_csv("literature_models.csv", dtype=str, keep_default_na=False)
vals = {k: s.value for k, s in sliders.items()}
E1, E3 = Engine(), Engine()
base_cost = E.run(vals)[0]; tb13 = wbv[S1]["B13"].value
cost_l1 = E1.run(vals, extra={(S1, "B13"): L1_G_DAY})[0]
cost_l3 = E3.run({**vals, (S1, "B9"): 0.645})[0]
# L2: tonnage for 1 K from the nine-model range and the ensemble sensitivity
K_PER_WM2 = 0.96 / 1.9
tg_lo, tg_hi = 7 / (2 * K_PER_WM2), 93 / (2 * K_PER_WM2)
surv = vals[(S6A, "B32")]; cpt = E.xl.evaluate(f"'{S6B}'!B6"); val_k = E.xl.evaluate(f"'{S6B}'!C21") * 1e9
bcr = lambda tg: val_k / (tg * 1e6 / surv * cpt)
rows = [
 ("L1", "Salt aerosol mass out per nozzle", f"Tito {S1.split('.')[0]}!B13 = B8*B9*B12*86400*1000", tb13, "g/day", L1_G_DAY, f"{100 * (L1_G_DAY / tb13 - 1):+.0f}%", "3a $/t aerosol", base_cost, cost_l1, f"{100 * (cost_l1 / base_cost - 1):+.0f}%"),
 ("L2 low end", "Aerosol for 1 K, lowest-emission model", "Tito 6a!G34:G37 (CESM2-Ghosh 2.43 .. UKESM1 99.7)", float(wbv[S6A]["G35"].value), "Tg/yr", tg_lo, f"{100 * (tg_lo / wbv[S6A]['G35'].value - 1):+.0f}% vs CESM2-Ghosh", "6b benefit-cost", tito[iB], bcr(tg_lo), "BCR at lit tonnage, Tito cost per tonne and survival"),
 ("L2 high end", "Aerosol for 1 K, highest-emission model", "Tito 6a!G34:G37", float(wbv[S6A]["G37"].value), "Tg/yr", tg_hi, f"{100 * (tg_hi / wbv[S6A]['G37'].value - 1):+.0f}% vs UKESM1", "6b benefit-cost", tito[iB], bcr(tg_hi), "BCR at lit tonnage, Tito cost per tonne and survival"),
 ("L3", "CCN-active fraction of sprayed particles", f"Tito {S1.split('.')[0]}!B9", 0.71, "fraction", 0.645, f"{100 * (0.645 / 0.71 - 1):+.0f}%", "3a $/t aerosol", base_cost, cost_l3, f"{100 * (cost_l3 / base_cost - 1):+.0f}%"),
 ("L4", "Survival to cloud base", "Tito 5a!B5 / 6a!B32 = 0.3", 0.3, "fraction", None, "no usable equation in held full text", "-", None, None, "see sweep in section D"),
]
D = pd.DataFrame(rows, columns=["id", "quantity", "Tito: origin", "Tito value", "unit", "literature value", "delta", "affects", "result, Tito chain", "result, with literature value", "delta result"])
display(D.style.format({"Tito value": "{:,.4g}", "literature value": "{:,.4g}", "result, Tito chain": "{:,.4g}", "result, with literature value": "{:,.4g}"}, na_rep="").hide(axis="index"))
display(LM[["id", "literature_method", "papers", "dois", "locator", "caveat"]].style.hide(axis="index").set_properties(**{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"}))
print("Both values are shown; Tito's formulas are not edited. L2 uses the preprint's 7-93 Tg/yr for -2 W/m2 and the ensemble sensitivity 0.96 K per 1.9 W/m2 (Stjern et al. 2018).")

## F. Conflicts and sources

In [ ]:
S = pd.read_csv("sources.csv", dtype=str, keep_default_na=False)
conf = R[R.conflict != ""][["sheet", "cell", "label", "conflict", "source_dois"]].copy()
conf["source_dois"] = conf.source_dois.apply(lambda s: " ".join(f"https://doi.org/{d}" for d in s.split()))
conf.loc[len(conf)] = ["(workbook-wide)", "-", "'~$10B/yr' deployment cost", "CONFLICT: the figure appears nowhere in the workbook (per-vessel $21.4M/yr; $175B-$18T/yr per K, tabs 6b/6c). Only a secondary, unsourced summary of one review gives ~$10B/yr.", "https://pmc.ncbi.nlm.nih.gov/articles/PMC6784395/"]
print("Conflicts between the workbook and the literature"); display(conf.style.hide(axis="index").set_properties(**{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top", "color": "#e0453a"}, subset=["conflict"]))
print("Sources"); S["doi_or_link"] = S.doi_or_link.apply(lambda s: f'<a href="{s}" target="_blank">{s}</a>' if s.startswith("http") else s)
display(HTML(S.to_html(escape=False, index=False)))

The two tables below are the added **Survey** (state of the art on the workbook's inputs and outputs, each number with a link, quote and flag) and **Summary** (the workbook's figures read against the survey). Flags: `stub` full text not available, `perplexity-only` only an AI-summary source, `hand-FX` currency converted by hand at the ECB rate, `unverified` quote not found, `no-source` left empty, `verified` read in the source.

In [ ]:
for name in ("survey", "summary"):
    d = pd.read_csv(f"{name}.csv", dtype=str, keep_default_na=False)
    print(name.capitalize(), f"({len(d)} rows)")
    link_cols = [c for c in d.columns if c.lower().startswith("source link")]
    for c in link_cols: d[c] = d[c].apply(lambda s: "<br>".join(f'<a href="{u}" target="_blank">{u.replace("https://doi.org/", "doi:")[:60]}</a>' for u in s.split()))
    flag = "Flag"
    d[flag] = d[flag].apply(lambda v: f'<b style="color:{"#3a9a4a" if v == "verified" else "#e0453a"}">{v}</b>')
    display(HTML(d.to_html(escape=False, index=False).replace("<td>", "<td style='text-align:left;vertical-align:top;white-space:pre-wrap'>")))

## G. Downloads

In [ ]:
import annotate_xlsx
def deliver(path):
    try:
        from google.colab import files; files.download(path)
    except ImportError:
        from IPython.display import FileLink; display(FileLink(path))
edits = {k: s.value for k, s in sliders.items() if not math.isclose(s.value, CENTRAL[k], rel_tol=1e-9)}
print("Edited inputs:", {lab[k]: v for k, v in edits.items()} or "none")
rd = lambda f: open(f, encoding="utf-8").read()
# 1. your edited copy of Tito's workbook (values only; formulas and charts untouched; Excel recalculates when opened)
EDITED = "mcb_tea_edited.xlsx"
annotate_xlsx.set_values(XLSX, EDITED, edits) if edits else shutil.copy(XLSX, EDITED)
# 2. optional: Tito's workbook with the literature notes beside the inputs and the Survey / Summary / Sources tabs, written from the CSVs at runtime
ANNOT = "mcb_tea_annotated.xlsx"
annotate_xlsx.annotate(EDITED, ANNOT, rd("ranges.csv"), rd("survey.csv"), rd("summary.csv"), rd("sources.csv"))
for f in (EDITED, ANNOT): print(f, os.path.getsize(f), "bytes")
deliver(ANNOT)